In [10]:
import os
import copy
import sys
import argparse
import asyncio
from tqdm import tqdm
import random
from pprint import pprint
from dotenv import load_dotenv

# Ensure repository root is on sys.path so "Code" is importable when run directly
REPO_ROOT = "/dartfs/rc/home/j/f006f3j/lab/omar/LoQA/"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Load environment variables from .env file
load_dotenv(os.path.join(REPO_ROOT, ".env"))

from Code.src.utils.io import save_json_file
from Code.src.utils.model_source import get_model
from Code.src.utils.prompts import (
    question_generation_prompt_template, 
    argument_extraction_prompt_template, 
    loqa_refinement_prompt_template,
)
from Code.src.utils.qg_and_pd_utils import (
    read_dataset_split_and_schema,
    get_valid_items,
    get_schema_question,
    get_response,
    process_qg_items_async,
)

In [11]:
dataset_root = '/dartfs/rc/home/j/f006f3j/lab/omar/LoQA/Dataset'
dataset_name = 'PHEE'
split_name = 'test'

qo_model_name = "gpt-oss-120b"
qo_model_origin = "dartmouth"
qo_model_access_string = "openai.gpt-oss-120b"
qo_model_reasoning_effort = "none"
qo_prompt_version = "v0"

pd_model_name = "gpt-oss-120b"
pd_model_origin = "dartmouth"
pd_model_access_string = "openai.gpt-oss-120b"
pd_model_reasoning_effort = "none"
pd_prompt_version = "zs-v0"

prompt_dir = "/dartfs/rc/home/j/f006f3j/lab/omar/LoQA/Prompts"
cache_dir = "/dartfs-hpc/rc/home/j/f006f3j/lab/shared"
initial_qo_prompt_version = "zs-v0"
num_samples = 100

dataset_split, dataset_schema = read_dataset_split_and_schema(dataset_root, dataset_name, split_name)

num_samples = len(dataset_split)
valid_items = get_valid_items(dataset_split, num_samples) #when I pass -1, it will return all the valid items
print(f"Number of valid items: {len(valid_items)}")

for item in valid_items[:5]:
    pprint(item)

Successfully loaded 14520 samples from HuggingFace
Reading JSON file from: /dartfs/rc/home/j/f006f3j/lab/omar/LoQA/Dataset/PHEE/PHEE-schema.json
Number of valid items: 4528
{'argument-type': 'main-arguments',
 'context': 'After therapy with parenteral amiodarone (2300 mg in 3 days) and '
            'other measures, signs of congestive heart failure disappeared; '
            'subsequently the patient developed jaundice, marked increase in '
            'serum transaminase levels and fall in prothrombin time, and '
            'histologic changes of severe centrilobular necrosis were observed '
            'in hepatic biopsy.',
 'event': 'adverse_event',
 'id': '3708949_1',
 'is-multi-event': False,
 'raw-initial-ground-truth': ['parenteral amiodarone (2300 mg in 3 days) and '
                              'other measures'],
 'role': 'treatment',
 'serial-number': 'test-2',
 'trigger': ['After']}
{'argument-type': 'main-arguments',
 'context': 'After therapy with parenteral amiodarone 

In [12]:
loqa_model = get_model(
    model_origin=qo_model_origin,
    model_access_string=qo_model_access_string,
    gen_temperature=0.0,
    gpu_uti=0.9,
    cache_dir=cache_dir,
    reasoning_effort=qo_model_reasoning_effort,
    hf_token=None,
)

pd_model1 = get_model(
    model_origin=pd_model_origin,
    model_access_string=pd_model_access_string,
    gen_temperature=0.0,
    gpu_uti=0.9,
    cache_dir=None,
    reasoning_effort=pd_model_reasoning_effort,
    hf_token=None,
)

initial_loqa_prompt_chain, _ = question_generation_prompt_template(loqa_model, prompt_file_path=os.path.join(prompt_dir, "qg", f"{initial_qo_prompt_version}.txt"))
arg_pd_chain, _ = argument_extraction_prompt_template(pd_model1, prompt_file_path=os.path.join(prompt_dir, "pd", f"{pd_prompt_version}.txt"))
loqa_opt_prompt_chain, loqa_opt_prompt_template = loqa_refinement_prompt_template(loqa_model, prompt_file_path=os.path.join(prompt_dir, "qo", f"{qo_prompt_version}.txt"))

Using Dartmouth Chat API model: openai.gpt-oss-120b
Model openai.gpt-oss-120b loaded successfully
Using Dartmouth Chat API model: openai.gpt-oss-120b
Model openai.gpt-oss-120b loaded successfully


In [13]:
from Code.src.utils.q_optimization_utils import run_refinement_loop_on_dataset

dataset = valid_items[:10]

results = run_refinement_loop_on_dataset(
    sampled_valid_items=dataset,
    initial_loqa_prompt_chain=initial_loqa_prompt_chain,
    arg_pd_chain=arg_pd_chain,
    refinement_prompt_chain=loqa_opt_prompt_chain,
    refinement_prompt_template=loqa_opt_prompt_template,
    get_response=get_response,
    num_iterations=5,
    target_score=1.0,
    maximum_patience=3,
    save_intermediate=False,
    debug_mode=True,
    output_path=None
)



Processing item 1/10

Generating initial questions...
{
  "questions": [
    "What medication was administered parenterally, including its total dose and the time frame over which it was given?",
    "What additional therapeutic measures were employed alongside the parenteral amiodarone treatment?"
  ]
}
Generated 2 initial questions

──────────────────────────────────────────────────────────────────────
Iteration 0
──────────────────────────────────────────────────────────────────────
Predicting arguments with 2 questions...
{
  "treatment": "parenteral amiodarone 2300 mg given over 3 days; other measures were employed alongside the parenteral amiodarone"
}
parenteral amiodarone 2300 mg given over 3 days || parenteral amiodarone 2300 mg in 3 days and other measures || 0.9740
Performance:
  F1:        0.67
  Precision: 0.50
  Recall:    1.00
  Matched:   1/1 GT args
  Predicted: 2 args (1 correct)
  New best! F1=0.67

Refining questions based on feedback...
{
  "questions": [
    "Wha

KeyError: 'serial_number'